# SAP-Ariba-Mock-Data-Generator-for-Procurement-Analytics


The report simulates a data structure originating from a hybrid procurement environment where the operational purchasing process is executed in the SAP Ariba cloud.

**Base Report:** SAP Ariba PO Report

**Business Purpose:** Generation of synthetic procurement data with a high degree of realism for the purpose of testing ETL pipelines and Machine Learning models designed to detect financial anomalies. Additionally, it serves as a dataset for procurement specialists and experts who want to practice data analysis and enhance their skills while maintaining compliance standards. All data is artificially created.



#### **Chart with description of columns, SAP Ariba rules and regex**

The dataset reflects the real SAP Ariba Report and its architecture.

In [11]:
import pandas as pd
import openpyxl

# wczytanie danych z pliku

df = pd.read_excel("/workspaces/SAP-Ariba-Mock-Data-Generator-for-Procurement-Analytics/data/mock_regex.xlsx")

In [12]:
# formatowanie nagłówka tabeli
naglowek = df.columns[0]

table_style = df.style.set_table_styles([
    {
        'selector':'th',
        'props':[
            ('font-weight', 'bold'),                #pogrubienie
            ('background-color', '#1e4974'),        #kolor tla
            ('color', 'white'),                     #kolor czcionki
            ('text-align', 'center'),               #wysrodkowanie 
            ('padding', '8px')
        ]
    }
]).set_properties(
    **{'font-weight':'bold'},                       #pogrubienie pierwszej kolumny
    subset=[naglowek]
).hide(axis='index')

table_style

Field,Business Name,Type and format,Example,Logic of generation and rules
PO Number,Purchase Order Number,Int (10 digits),6000026225,Always starts with 6000 and has 10 digits. Unique.
Company Code,Company Code,String (4 letters/numbers),A001,Organizational unit code. Values taken from a predefined company dictionary.
Supplier ID,Supplier Identifier,Int (10 digits),2008890021,Always starts with 200 and has 10 digits. Unique per supplier.
Supplier Name,Supplier Name,String (up to 50 chars),Nimbus,Random company name selected from supplier list.
Requester ID,Requester/User ID,String (8 letters),PLTOZIE,8 characters: “PL” + first 3 letters of first name + first 3 letters of last name.
Requester Name,Requester Full Name,String (up to 50 chars),Tomasz Zieliński,Generated based on Requester ID.
Requester Mail,Requester Email Address,String (email format),tomasz.zielinski@firma.com,First and last name converted to email format + company domain.
Create Date,Create Date,Date (dd.mm.yyyy),2026-03-25 00:00:00,Random date within the time frame
Delivery Date,Delivery Date,Date (dd.mm.yyyy),2026-03-25 00:00:00,Random date within ±90 days from invoice date.
Order Status,Purchase Order Status,"Enum (received, invoiced, pending ap, etc.)",received,Value selected from the list of PO process statuses.


*In practice, both numeric codes (e.g., 0030 — meaning 30 days net from the invoice issue date) and alphanumeric codes (e.g., E30M — end of month plus 30 days) are used. The choice of which codes are applied in SAP Ariba depends on architectural decisions made by the IT and Finance departments during implementation. In one company, “30 days net from invoice receipt” may be coded as 0030, while in another it may appear as 030N or N30. The choice is also influenced by whether the organization uses the traditional Ariba environment or has already migrated to S/4HANA — in newer versions, consultants often recommend using alphanumeric codes.


In my dataset, however, I focus on numeric codes because this is the environment I previously worked in and am most familiar with. Additionally, this approach allows users who rely on the dataset for learning purposes to concentrate on data analysis itself, without needing to interpret the meaning behind different payment term codes.


#### **Technical Notes & Architecture (Business Rules Callouts):**

1.	**Currency consistency rule** — both the purchase order and the invoice must use the same currency, e.g., EUR, CHF, or PLN.

2.	Status logic --> Purchase Order (PO) Status:

    - **ordered** — means that the purchase order has been created and sent to the supplier.

    - **confirmed** — means that the supplier has received the order and confirmed its execution. This status is very often skipped in real life operations.
        
    - **received** — means that the delivery has been physically received by the requester, and the quantity and quality of the goods have been verified. This status is the basis for starting the invoicing process.

    - **invoiced** — means that the supplier has issued an invoice for the delivered goods or services, and the document has been registered in the system.
    
    - **paid** - (optional, if supported by the system) — means that the invoice has been paid according to the payment terms, and the purchase order process is completed.

4. **Artificial errors:** the script intentionally introduces incorrect data — for example empty values or price discrepancies — in order to simulate a real business environment.


#### **Status Workflow – logic of Purchase Order Status Change**


1. ordered → confirmed

The purchase order has been created and sent to the supplier. If the supplier confirms receipt of the order, the status changes to confirmed. The script should first generate the status ordered, and then — optionally — confirmed.

2. confirmed → received

The goods or services have been delivered and received by the requester. The status received indicates that the delivery has been physically confirmed. The script should enforce that received can only appear after ordered or confirmed.

3. received → invoiced

After the delivery has been received, the supplier issues an invoice. The status invoiced indicates that the document has been registered in the system. The script must ensure that invoiced cannot appear before received.

4. invoiced → paid (jeśli występuje w Twoim modelu)

The invoice has been paid, and the purchase order is fully settled. The script should end the workflow at paid and prevent any further status changes.

![Diagram PO Status change](/workspaces/SAP-Ariba-Mock-Data-Generator-for-Procurement-Analytics/images/po_status.png)

Purchase order status — to reflect real business logic, the script only refers to orders with the status received. Why? Because received confirms the order amount, the quantity of items, and the physical receipt of the goods or services. If the buyer notices that something on the invoice does not match, they should clarify the issue and adjust the purchase order and the goods receipt (GR) to reflect the actual situation.

#### **Edge Cases & Expectations: outliers, empty values, price variance**


In the dataset, I intentionally included:

- outliers — 0.04%

- empty values in the delivery date — random, below 2%

- discrepancies between the purchase order amount and the invoice amount — up to 12% of all records

- weighting (percentage values) for purchase order statuses, invoice statuses, and amount ranges*


#### **Table: Identifiers and data formats (Date Masking / Formats)**

In [3]:
df2 = pd.read_excel('/workspaces/SAP-Ariba-Mock-Data-Generator-for-Procurement-Analytics/data/regex1.xlsx')

table_style = df2.style.set_table_styles([
    {
        'selector':'th',
        'props':[
            ('font-weight', 'bold'),                #pogrubienie
            ('background-color', '#1e4974'),        #kolor tla
            ('color', 'white'),                     #kolor czcionki
            ('text-align', 'center'),               #wysrodkowanie 
            ('padding', '8px')
        ]
    }
]).set_properties(
    **{'font-weight':'bold'},                       #pogrubienie pierwszej kolumny
    subset=[naglowek]
).hide(axis='index')

table_style

Field,Description,Regex / Pattern
Vendor ID,"It always begins with the prefix 200, followed by 7 digits.",^200[0-9]{7}$
PO Number,It always starts with 6000 and has a fixed length of 10 digits.,^6000[0-9]{6}$
Invoice ID,"System prefix INV-, followed by 2 digits + /4 digits.",^INV-[0-9]{2}/[0-9]{4}$
Date (ISO 8601),Date format: YYYY‑MM‑DD.,^\d{4}-\d{2}-\d{2}$
Amount,"A number with 2 decimal places, using the dot as the separator.",^\d+\.\d{2}$
Currency,Currency code in ISO 4217 format (3 letters).,^[A-Z]{3}$
Payment Term,An integer value (days).,^\d+$


#### **Conclusions**

This notebook summarizes the core business logic, validation rules, regex patterns, and data‑generation workflow used to create realistic, GDPR‑safe synthetic procurement data aligned with SAP Ariba standards. The approach enables fast, scalable creation of high‑quality datasets suitable for analytics, testing, dashboards, and training.


💡 Key Takeaways:

- Synthetic data follows real procurement processes and approval flows.

- Regex validation ensures consistent and reliable field formats.

- The generator produces large datasets in seconds and is easy to extend.

- Output files are ready for immediate use in Power BI, SQL, Python, or Excel.


✨ Future Improvements:

- Add modules for invoices, contracts, and sourcing events.

- Extend validation rules and business logic for additional fields.

- Introduce more pytest tests
